# 机组排班（Beasley-Cao csp50）—— 直接建模（完整池集合覆盖 IP）

## 问题定义

50 个任务 $i$（固定起止时间 $s_i,f_i$）；时间上限 $T=480$；173 条转移弧 $(i,j,c_{ij})$。
一个 crew 的任务序列须逐对由弧连接（弧列表已编码时间兼容性）且**跨度** $f_{last}-s_{first}\le 480$。
目标：**最少 crew 数，其次最小总转移成本**。集合覆盖模型：

$$\min_x \sum_{p\in P} c_p x_p \quad \text{s.t.}\quad \sum_{p\in P} a_{ip}x_p \ge 1\ (\forall i),\quad \sum_{p\in P} x_p \le K,\quad x_p\in\{0,1\}$$

列 $p$ = 一条可行 crew 调度（任务序列），$c_p$ = 序列转移成本之和。
**基准最优**：27 crew、成本 3139（本家族 01 直接模型证明，K=26 不可行）。


In [1]:
# -*- coding: utf-8 -*-
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import math, time, datetime
from ortools.math_opt.python import mathopt
from ortools.sat.python import cp_model

DATA = "/mnt/d/exactTest/column-generation-testcases/crew_scheduling/csp50.txt"
_lines = open(DATA).read().splitlines()
N, T = map(int, _lines[0].split())
tasks = [None] + [tuple(map(int, l.split())) for l in _lines[1:1+N]]
arc_cost = {}
for l in _lines[1+N:]:
    i, j, c = map(int, l.split()); arc_cost[(i, j)] = c
K_MIN = 27
EPS = 1e-7
M_DUMMY = 10**6

def enumerate_pool():
    adj = {}
    for (i, j), c in arc_cost.items():
        adj.setdefault(i, []).append((j, c))
    paths = []
    for start in range(1, N+1):
        s0 = tasks[start][0]
        stack = [(start, [start], 0)]
        while stack:
            u, seq, c = stack.pop()
            paths.append((tuple(seq), c, tasks[u][1]-s0))
            for v, cv in adj.get(u, []):
                span = tasks[v][1] - s0
                if span <= T:
                    stack.append((v, seq+[v], c+cv))
    return paths

paths = enumerate_pool()
P = len(paths)
pseq = [p[0] for p in paths]
pcost = [p[1] for p in paths]
pmask = []
for s2 in pseq:
    m2 = 0
    for i in s2:
        m2 |= (1 << i)
    pmask.append(m2)


print(f"完整池: {P} 条可行路径（单任务 {sum(1 for s in pseq if len(s)==1)} / 双 {sum(1 for s in pseq if len(s)==2)} / 三 {sum(1 for s in pseq if len(s)==3)}）")
print(f"下界: 时长={math.ceil(sum(f-s for s,f in tasks[1:])/T)} | 已知最优: 27 crew / 3139（K=26 不可行）")


完整池: 266 条可行路径（单任务 50 / 双 173 / 三 43）
下界: 时长=14 | 已知最优: 27 crew / 3139（K=26 不可行）


## 方法：直接建模基准

- **模型**：完整池上的集合覆盖整数规划（HIGHS）：池=全部 266 条可行 crew 调度（DFS 完全枚举，
  跨度≤480 与弧兼容性检查 ⇒ 与原问题**等价**），故这是原问题的精确整数模型：
  $$\min\sum_p c_px_p\ \ \text{s.t.}\ \sum_p a_{ip}x_p\ge 1,\ \sum_p x_p\le K,\ x_p\in\{0,1\}$$
- **最少 crew 数证明**：K=26 不可行 + K=27 最优（0.02s）；容量/重叠/入度下界（14/10/10）仅作参考。
- CP-SAT 交叉验证同解；说明：紧凑 3-index CP-SAT 草稿（direct_solve.py）在 K=16 即 30s 未决，
  故基准采用等价池模型。


In [2]:
def col_arcs(seq):
    a = []
    prev = 0
    for j in seq:
        a.append((prev, j))
        prev = j
    a.append((prev, N+1))
    return a

def pool_ip(Kv, use_cpsat=False, time_limit=60.0):
    if not use_cpsat:
        m = mathopt.Model()
        x = [m.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"x{p}") for p in range(P)]
        for i in range(1, N+1):
            m.add_linear_constraint(mathopt.fast_sum([x[p] for p in range(P) if (pmask[p] >> i) & 1]) >= 1.0, name=f"c{i}")
        m.add_linear_constraint(mathopt.fast_sum(x) <= Kv, name="veh")
        m.minimize(mathopt.fast_sum([pcost[p]*x[p] for p in range(P)]))
        t0 = time.time()
        res = mathopt.solve(m, mathopt.SolverType.HIGHS,
                            params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=time_limit), enable_output=False))
        wt = time.time() - t0
        obj = res.objective_value() if res.termination.reason in (mathopt.TerminationReason.OPTIMAL, mathopt.TerminationReason.FEASIBLE) else None
        sel = [paths[p] for p in range(P) if res.variable_values()[x[p]] > 0.5] if obj is not None else None
        return res.termination.reason.name, obj, sel, wt
    else:
        m = cp_model.CpModel()
        x = [m.NewBoolVar(f"x{p}") for p in range(P)]
        for i in range(1, N+1):
            m.Add(sum(x[p] for p in range(P) if (pmask[p] >> i) & 1) >= 1)
        m.Add(sum(x) <= Kv)
        m.Minimize(sum(pcost[p]*x[p] for p in range(P)))
        s = cp_model.CpSolver()
        s.parameters.max_time_in_seconds = time_limit
        t0 = time.time()
        st = s.Solve(m)
        wt = time.time() - t0
        sel = [paths[p] for p in range(P) if s.Value(x[p])] if st in (cp_model.OPTIMAL, cp_model.FEASIBLE) else None
        return s.StatusName(st), (s.ObjectiveValue() if st in (cp_model.OPTIMAL, cp_model.FEASIBLE) else None), sel, wt

def run_direct(verbose=True):
    total_dur = sum(f-s for s, f in tasks[1:])
    ev = []
    for s, f in tasks[1:]:
        ev.append((s, 1)); ev.append((f, -1))
    ev.sort(key=lambda x: (x[0], -x[1]))
    cur = overlap = 0
    for t2, d in ev:
        cur += d; overlap = max(overlap, cur)
    has_in = {j for (i, j) in arc_cost}
    noin = [i for i in range(1, N+1) if i not in has_in]
    if verbose:
        print(f"下界: 时长={math.ceil(total_dur/T)} 重叠={overlap} 无入弧={len(noin)} | 池 {P} 条路径")
    term26, obj26, _, wt26 = pool_ip(26)
    term27, obj27, routes, wt27 = pool_ip(27)
    term27c, obj27c, _, wt27c = pool_ip(27, use_cpsat=True)
    if verbose:
        print(f"K=26: {term26}（{round(wt26,3)}s）-> 最少 crew = 27")
        print(f"K=27 HIGHS: {term27} obj={obj27}（{round(wt27,3)}s）")
        print(f"K=27 CP-SAT 交叉验证: {term27c} obj={obj27c}（{round(wt27c,2)}s）")
        for r in sorted(routes, key=lambda s: -len(s)):
            print("  crew:", r, "成本", sum(arc_cost.get((r[k], r[k+1]), 0) for k in range(len(r)-1)))
    return dict(K_min=27, objective=obj27, routes=routes, term26=term26, term27=term27)

res = run_direct(verbose=True)


下界: 时长=14 重叠=10 无入弧=10 | 池 266 条路径
K=26: INFEASIBLE（0.018s）-> 最少 crew = 27
K=27 HIGHS: OPTIMAL obj=3139.0（0.012s）
K=27 CP-SAT 交叉验证: OPTIMAL obj=3139.0（0.05s）
  crew: ((1, 10), 169, 399) 成本 0
  crew: ((2, 13), 155, 430) 成本 0
  crew: ((3,), 0, 152) 成本 0
  crew: ((4,), 0, 21) 成本 0
  crew: ((5, 19), 77, 373) 成本 0
  crew: ((6, 15), 130, 438) 成本 0
  crew: ((7, 20), 307, 463) 成本 0
  crew: ((8, 17), 91, 318) 成本 0
  crew: ((9, 16, 23), 373, 473) 成本 0
  crew: ((11, 22), 26, 460) 成本 0
  crew: ((12, 27), 83, 407) 成本 0
  crew: ((14, 24), 22, 405) 成本 0
  crew: ((18, 25, 31), 257, 351) 成本 0
  crew: ((21, 32, 35), 341, 466) 成本 0
  crew: ((26, 33), 104, 308) 成本 0
  crew: ((28,), 0, 59) 成本 0
  crew: ((29, 34), 226, 295) 成本 0
  crew: ((30, 36), 151, 430) 成本 0
  crew: ((37, 44), 182, 366) 成本 0
  crew: ((38, 42), 218, 448) 成本 0
  crew: ((39, 41), 68, 347) 成本 0
  crew: ((40, 46), 132, 277) 成本 0
  crew: ((43,), 0, 49) 成本 0
  crew: ((45,), 0, 127) 成本 0
  crew: ((47, 50), 27, 195) 成本 0
  crew: ((48,), 0, 203) 

## 运行结果与结论

见上方输出。结论：

- **K=26 不可行 ⇒ 最少 27 crew；K=27 最优成本 3139（HIGHS 与 CP-SAT 交叉验证一致），作为家族基准。**
- 基准最优值来源：本家族 01_direct 证明（K=26 不可行 + K=27 最优 3139）；
  各方法结果交叉一致。
